# Next-Word Prediction: Preprocessing

This notebook walks the pipeline from the design sketch, one stage at a time:

`text` -> `clean` -> `tokenize` -> `token ids` -> `sliding windows (context -> next word)`

It ends by saving the processed splits into `data/processed/` and the vocabulary into `models/tokenizer.json`, the file the web app loads at inference time.

Run `01_eda.ipynb` first if you have not seen the data yet. Because the corpus mixes Twitter text in, `clean_text` also strips URLs, @mentions, and leading "RT @user:" noise, and unwraps hashtags to plain words — none of which plain prose has.

## 1. Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(".."))  # so `import config` and `from src...` work from notebooks/

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src.dataset import load_corpus, prepare_data
from src.preprocess import Vocabulary, clean_text, split_sentences, tokenize

np.random.seed(config.RANDOM_STATE)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

print("Corpus:", config.DEFAULT_CORPUS)
print("Outputs:", config.OUTPUTS_DIR)

Corpus: D:\next-word-prediction\data\raw\corpus.txt
Outputs: D:\next-word-prediction\outputs


## 2. One sentence, step by step

In [2]:
example = "I want to go to the"
cleaned = clean_text(example)
tokens = tokenize(example)
print("clean    :", cleaned)
print("tokenize :", tokens)

clean    : i want to go to the
tokenize : ['i', 'want', 'to', 'go', 'to', 'the']


## 3. Build the vocabulary (training sentences only)

Sentences are split into train, validation and test **before** the vocabulary is built. Fitting the vocabulary on all text would leak test words into training.

In [3]:
from src.dataset import split_sentence_list

sentences = split_sentences(load_corpus())
train_s, val_s, test_s = split_sentence_list(sentences, config.VAL_FRACTION, config.TEST_FRACTION, config.RANDOM_STATE)
print(f"train {len(train_s):,} | val {len(val_s):,} | test {len(test_s):,} sentences")

vocab = Vocabulary.build(train_s)
print("vocabulary size:", len(vocab))
print("first 15 ids   :", list(enumerate(vocab.itos[:15])))

train 622,255 | val 77,781 | test 77,781 sentences
vocabulary size: 10000
first 15 ids   : [(0, '<pad>'), (1, '<unk>'), (2, 'the'), (3, 'to'), (4, 'and'), (5, 'a'), (6, 'of'), (7, 'in'), (8, 'i'), (9, 'that'), (10, 'for'), (11, 'is'), (12, 'it'), (13, 'on'), (14, 'you')]


In [4]:
ids = vocab.encode(tokens)
print("tokens   :", tokens)
print("token ids:", ids)
print("decoded  :", vocab.decode(ids))

tokens   : ['i', 'want', 'to', 'go', 'to', 'the']
token ids: [8, 118, 3, 98, 3, 2]
decoded  : ['i', 'want', 'to', 'go', 'to', 'the']


## 4. Sliding windows

Each sentence becomes many (previous words -> next word) pairs. Contexts shorter than `SEQ_LEN` are left padded with `<pad>` (id 0), which the model's embedding layer masks.

In [5]:
from src.dataset import make_windows

x_demo, y_demo = make_windows([vocab.encode(tokens)], seq_len=config.SEQ_LEN)
for row, target in zip(x_demo, y_demo):
    print(f"{vocab.decode(row)}  ->  {vocab.itos[target]}")

['<pad>', '<pad>', '<pad>', '<pad>', '<pad>', 'i']  ->  want
['<pad>', '<pad>', '<pad>', '<pad>', 'i', 'want']  ->  to
['<pad>', '<pad>', '<pad>', 'i', 'want', 'to']  ->  go
['<pad>', '<pad>', 'i', 'want', 'to', 'go']  ->  to
['<pad>', 'i', 'want', 'to', 'go', 'to']  ->  the


## 5. Build all splits

In [6]:
splits = prepare_data(load_corpus())
pd.Series(splits.stats())

vocab_size         10000
train_samples    9848326
val_samples      1227410
test_samples     1223948
seq_len                6
dtype: int64

In [7]:
for i in np.random.default_rng(0).choice(len(splits.y_train), 8, replace=False):
    context = [w for w in splits.vocab.decode(splits.x_train[i]) if w != config.PAD_TOKEN]
    print(f"{' '.join(context):<40} -> {splits.vocab.itos[splits.y_train[i]]}")

the batter into the pan and              -> slide
the following plants that attract <unk>  -> what
san jose the blues have to               -> prevent
slice of cheese in my <unk>              -> when
among the crowd                          -> at
make that offer all we try               -> to
the hired party the extent of            -> the
the air force almost a year              -> ago


In [8]:
assert splits.x_train.shape[1] == config.SEQ_LEN
assert splits.x_train.max() < len(splits.vocab) and splits.y_train.max() < len(splits.vocab)
assert splits.x_train.min() >= 0
print("Shape and range checks passed")

Shape and range checks passed


## 6. Class balance

Next-word prediction is extremely imbalanced: a few words (`the`, `and`, `to`) dominate. This is why we report top-5 accuracy and perplexity, not only top-1.

In [9]:
unk_share = (splits.y_train == config.UNK_ID).mean()
top_share = pd.Series(splits.y_train).value_counts(normalize=True).head(5)
print(f"Target is <unk> in {unk_share:.1%} of training samples")
pd.DataFrame({"word": [splits.vocab.itos[i] for i in top_share.index], "share of targets": top_share.round(4).values})

Target is <unk> in 7.9% of training samples


,word,share of targets
0,<unk>,0.0790
1,the,0.0495
2,to,0.0292
3,and,0.0265
4,a,0.0253


## 7. Save processed data and the tokenizer

In [10]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name in ("train", "val", "test"):
    np.savez_compressed(config.PROCESSED_DIR / f"{name}.npz", x=getattr(splits, f"x_{name}"), y=getattr(splits, f"y_{name}"))
(config.PROCESSED_DIR / "stats.json").write_text(json.dumps(splits.stats(), indent=2))
splits.vocab.save(config.VOCAB_PATH)
print("Saved:", sorted(p.name for p in config.PROCESSED_DIR.glob("*")), "and", config.VOCAB_PATH)

Saved: ['.gitkeep', 'stats.json', 'test.npz', 'train.npz', 'val.npz'] and D:\next-word-prediction\models\tokenizer.json


Next: `03_lstm.ipynb`.